# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kubrafathima2005-kf/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: The Anatomy of Growing Content

The research reports that growing pages were younger on average (185 days) than declining pages (228 days). Their average word counts were almost the same.

**My methodology question:** How were growing and declining pages labelled? Which time periods were compared to decide whether a page was growing or declining? Knowing this helps us understand where the result comes from and whether the label is reliable.

### Finding 2: Growth Prediction Model

The research reports about 90% accuracy on unseen pages from known brands and about 75% on completely unseen brands.

**My methodology question:** How were the training and testing groups separated? Were pages from unseen brands kept completely separate during testing? This helps us understand whether the model can generalise to new brands and clients.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Load dataset
url = "https://raw.githubusercontent.com/kubrafathima2005-kf/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Use metadata features only
numeric_features = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count"
]

categorical_features = [
    "content_type", "main_intent", "competition_level"
]

features = numeric_features + categorical_features
X = df[features]
y = df["is_declining_label"]

# Prepare model
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

def make_model():
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced"
        ))
    ])

def precision_at_50(model, train_idx, test_idx):
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = model.predict_proba(X.iloc[test_idx])[:, 1]
    top_50 = y.iloc[test_idx].iloc[
        scores.argsort()[::-1][:50]
    ]
    return top_50.mean()

# BEFORE: Random split
train_random, test_random = train_test_split(
    df.index, test_size=0.2, random_state=42,
    stratify=y
)

random_model = make_model()
random_p50 = precision_at_50(
    random_model, train_random, test_random
)

# AFTER: Grouped split by client
splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=42
)

train_group, test_group = next(
    splitter.split(X, y, groups=df["client_id"])
)

group_model = make_model()
group_p50 = precision_at_50(
    group_model, train_group, test_group
)

print("Random split Precision@50:", round(random_p50, 3))
print("Grouped split Precision@50:", round(group_p50, 3))
print("Grouped training clients:",
      df.iloc[train_group]["client_id"].nunique())
print("Grouped testing clients:",
      df.iloc[test_group]["client_id"].nunique())
print("Overall positive label rate:", round(y.mean(), 3))

Random split Precision@50: 0.74
Grouped split Precision@50: 0.4
Grouped training clients: 25
Grouped testing clients: 7
Overall positive label rate: 0.542


### Results and Interpretation

The random split achieved a Precision@50 of 0.74, while the grouped client split achieved 0.40.

This shows a decrease of 0.34 when testing on unseen clients. The result suggests that the model performs less consistently on new clients.

The overall positive label rate was 0.542. Therefore, the grouped result was below the overall positive rate. These results are directional and show why grouped validation is important.

### Before and After Validation

Before: I will use a random 80/20 train-test split. Pages from the same client may appear in both sets.

After: I will use a grouped 80/20 split by client. Each client will appear in either training or testing, not both.

I will compare both models using Precision@50. The grouped split gives a more realistic check of performance on unseen clients.

For this audit, I will use content and keyword metadata only. I will exclude trend labels and overlapping performance-window features.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage Audit

I checked the final model features for possible data leakage.

I excluded trend_direction, trend_pct, and is_declining_label because they directly define the target. I also excluded recent and previous performance-window metrics because they may overlap with the period used to create the target.

The final feature set uses content and keyword metadata only. This reduces the risk of using information that would not be available at prediction time.

In [6]:
excluded_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print("Final model features:")
print(features)

print("\nExcluded leakage or identifier columns:")
print(excluded_columns)

print("\nTarget included in features:", "is_declining_label" in features)
print("Trend direction included:", "trend_direction" in features)
print("Trend percentage included:", "trend_pct" in features)

Final model features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_type', 'main_intent', 'competition_level']

Excluded leakage or identifier columns:
['trend_direction', 'trend_pct', 'is_declining_label', 'client_id', 'content_id', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Target included in features: False
Trend direction included: False
Trend percentage included: False


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original Claim

My Random Forest model achieved 52% accuracy and 0.52 Precision@50, so it can predict declining content effectively.

### Rewritten Safe Claim

In my experiment, the Random Forest model achieved a Precision@50 of 0.52 under the original evaluation and 0.40 under grouped client validation. The lower grouped result suggests that performance may not transfer well to unseen clients. The model can currently be considered a decision-support experiment, not a proven prediction system.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Self-check

1. I included two research findings and constructive methodology questions.
2. I compared random and grouped client validation.
3. I checked the final features for possible data leakage.
4. I excluded target labels, identifiers, and overlapping performance metrics.
5. I rewrote my claims using careful and honest language.
6. I used only anonymized data without client names, URLs, or private queries.

The grouped validation result was lower than the random split result, showing the importance of honest validation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.